In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

In [ ]:
seawulf_df = gpd.read_file('../output/Georgia/ga_seawulf.gpkg')
nyt_df = gpd.read_file("../../data/Georgia/Precinct/ga_nyt.geojson")

In [3]:
print(f"seawulf_df: {seawulf_df.shape}")
print(f"nyt: {nyt_df.shape}")

seawulf_df: (2720, 11)
nyt: (2680, 8)


In [4]:
seawulf_df.head()

,UNIQUE_ID,Kamala D. Harris,Donald J. Trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,geometry
0,APPLING-:-1B,108,921,2,1031,1074.216587,102.982480,55.003760,3.618320,1235.821146,"MULTIPOLYGON (((375680.672 3535246.721, 375706..."
1,APPLING-:-1C,67,724,0,791,1191.912652,472.673691,42.287709,28.713142,1735.587193,"MULTIPOLYGON (((378693.089 3524550.847, 378692..."
2,APPLING-:-2,782,541,4,1327,1235.612661,788.224241,96.849558,32.601383,2153.287844,"MULTIPOLYGON (((380005.78 3525000.183, 380113...."
3,APPLING-:-3A1,31,617,0,648,705.543283,121.434717,146.434230,7.825296,981.237527,"MULTIPOLYGON (((377772.229 3534595.104, 377753..."
4,APPLING-:-3C,246,934,3,1183,1427.934438,416.186299,57.464584,60.117966,1961.703287,"MULTIPOLYGON (((386430.886 3520109.322, 386469..."


In [5]:
zero_votes = seawulf_df[seawulf_df['Total_votes']==0]
zero_votes.shape

(26, 11)

In [6]:
nyt_df.head()

,state,GEOID,votes_dem,votes_rep,votes_total,pct_dem_lead,official_boundary,geometry
0,GA,13001-4B,25,720,745,-0.9329,False,"POLYGON ((-82.31151 31.59099, -82.31149 31.590..."
1,GA,13001-5B,82,643,727,-0.7717,False,"POLYGON ((-82.40153 31.69729, -82.40206 31.697..."
2,GA,13001-3C,246,934,1183,-0.5816,False,"POLYGON ((-82.13346 31.6308, -82.13346 31.6308..."
3,GA,13001-2,782,541,1327,0.1816,False,"POLYGON ((-82.28049 31.84937, -82.2805 31.8493..."
4,GA,13001-1B,108,921,1031,-0.7886,False,"POLYGON ((-82.29335 31.94084, -82.29344 31.940..."


In [7]:
zero_votes_nyt = nyt_df[nyt_df['votes_total']==0]
zero_votes_nyt.shape

(0, 8)

In [29]:
diff = (seawulf_df.shape[0] - nyt_df.shape[0]) - zero_votes.shape[0]
print(f"there are {diff} more precicnts in seawulf")

there are 14 more precicnts in seawulf


In [9]:
zero_votes = zero_votes['geometry']

In [10]:
zero_votes_nyt = zero_votes_nyt['geometry']

In [11]:
print(zero_votes.crs)
print(zero_votes_nyt.crs)

EPSG:32617
EPSG:4326


In [12]:
zero_votes = zero_votes.to_crs(epsg=4326)

In [13]:
zero_votes.head()

122    MULTIPOLYGON (((-81.40485 31.93704, -81.40496 ...
312    MULTIPOLYGON (((-81.02006 32.09349, -81.02002 ...
313    MULTIPOLYGON (((-84.82498 32.41361, -84.82476 ...
314    MULTIPOLYGON (((-84.79455 32.30975, -84.79436 ...
315    MULTIPOLYGON (((-84.95734 32.30818, -84.95747 ...
Name: geometry, dtype: geometry

In [15]:
print(zero_votes.crs)
print(zero_votes_nyt.crs)

EPSG:4326
EPSG:4326


In [16]:
zero_votes_nyt.is_valid.value_counts()

Series([], Name: count, dtype: int64)

In [17]:
zero_votes_nyt = zero_votes_nyt[~zero_votes_nyt.is_empty]

In [19]:
print(zero_votes_nyt.total_bounds)
print(zero_votes_nyt.head())

[nan nan nan nan]
GeoSeries([], Name: geometry, dtype: geometry)


In [22]:
import matplotlib.pyplot as plt

In [28]:
zero_votes = gpd.GeoDataFrame(zero_votes, geometry="geometry")
zero_votes = zero_votes.to_crs(nyt_df.crs)
# spatial join: zero_votes → nyt_df
joined = gpd.sjoin(
    zero_votes,
    nyt_df,
    how="inner",
    predicate="intersects"   # or "within" if strictly inside
)

print(joined.shape)

(75, 9)


In [30]:
joined

,geometry,index_right,state,GEOID,votes_dem,votes_rep,votes_total,pct_dem_lead,official_boundary
122,"MULTIPOLYGON (((-81.40485 31.93704, -81.40496 ...",2043,GA,13179-06,606,1226,1841,-0.3368,False
122,"MULTIPOLYGON (((-81.40485 31.93704, -81.40496 ...",121,GA,13029-04,1146,1621,2785,-0.1706,False
122,"MULTIPOLYGON (((-81.40485 31.93704, -81.40496 ...",236,GA,13051-6-11C,2202,1151,3382,0.3108,False
122,"MULTIPOLYGON (((-81.40485 31.93704, -81.40496 ...",2045,GA,13179-13,1047,698,1760,0.1983,False
122,"MULTIPOLYGON (((-81.40485 31.93704, -81.40496 ...",122,GA,13029-01,598,1215,1819,-0.3392,False
...,...,...,...,...,...,...,...,...,...
1633,"MULTIPOLYGON (((-84.51585 33.58683, -84.51593 ...",1384,GA,13121-SC09B,1206,70,1284,0.8847,True
1633,"MULTIPOLYGON (((-84.51585 33.58683, -84.51593 ...",1539,GA,13121-SC08B,1987,125,2126,0.8758,True
1645,"MULTIPOLYGON (((-84.50794 33.58321, -84.50832 ...",1385,GA,13121-UC031,1379,228,1616,0.7123,True
1645,"MULTIPOLYGON (((-84.50794 33.58321, -84.50832 ...",1384,GA,13121-SC09B,1206,70,1284,0.8847,True
